# 03 — Silver Layer
**Purpose:** clean, rename, type-check, and derive — produce the source-of-truth table for downstream consumers.
**Input:** `workspace.taxi.bronze_yellow_taxi`
**Output:** `workspace.taxi.silver_yellow_taxi`
**Transforms applied:**
- Snake_case rename
- Drop rows with null pickup_datetime / dropoff_datetime / fare_amount / passenger_count
- Compute `trip_duration_minutes`, `pickup_date`
- (Validation rules + rejected_records → next notebook)


In [0]:
%run ./00_utils

In [0]:
import uuid
RUN_ID = str(uuid.uuid4())

Loaded helpers: LOG_TABLE, LOG_SCHEMA, log_pipeline_run, PIPELINE_NAME


### The validation pattern

Rather than dropping bad rows, we tag each row with the names of any rules it failed (`validation_failures` array column). Rows with zero failures go to `silver_yellow_taxi`; rows with one or more failures go to `rejected_records`, preserving every source column.

This makes failures **recoverable** — once an upstream rule is fixed, the affected rows can be re-processed from quarantine. Dropping would have hidden the data quality issue and made backfill impossible.

For January 2024 this surfaced ~68k quarantined rows: mostly `fare_not_positive` (refund codes / $0 trips) and `passenger_count_out_of_range` (drivers logging 0 passengers on real trips). The latter category is *recoverable revenue* — real trips happened, the data is just imperfect.

In [0]:
from pyspark.sql import functions as F

BRONZE_TABLE   = "workspace.taxi.bronze_yellow_taxi"
SILVER_TABLE   = "workspace.taxi.silver_yellow_taxi"
REJECTED_TABLE = "workspace.taxi.rejected_records"

try:
    df_bronze = spark.table(BRONZE_TABLE)
    rows_in = df_bronze.count()

    # 1. Snake_case rename
    rename_map = {
        "VendorID":             "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime":"dropoff_datetime",
        "RatecodeID":           "ratecode_id",
        "PULocationID":         "pickup_location_id",
        "DOLocationID":         "dropoff_location_id",
        "Airport_fee":          "airport_fee",
    }
    df = df_bronze
    for old, new in rename_map.items():
        if old in df.columns:
            df = df.withColumnRenamed(old, new)

    # 2. Drop structurally-required nulls (can't validate what doesn't exist)
    df = df.dropna(subset=["pickup_datetime", "dropoff_datetime", "fare_amount", "passenger_count"])

    # 3. Derive computed columns
    df = (df
        .withColumn("trip_duration_minutes",
                    (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60.0)
        .withColumn("pickup_date", F.to_date("pickup_datetime"))
    )

    # 4. Apply business validation rules — collect failure names per row
    df = df.withColumn(
        "validation_failures",
        F.array_compact(F.array(
            F.when(~(F.col("fare_amount") > 0),                                          F.lit("fare_not_positive")),
            F.when(~((F.col("passenger_count") >= 1) & (F.col("passenger_count") <= 6)), F.lit("passenger_count_out_of_range")),
            F.when(~(F.col("trip_distance") >= 0),                                       F.lit("negative_distance")),
            F.when(~(F.col("pickup_datetime") < F.col("dropoff_datetime")),              F.lit("non_chronological_times")),
        ))
    )

    # 5. Split — clean rows go to Silver, failed rows go to rejected_records
    df_clean    = df.filter(F.size("validation_failures") == 0).drop("validation_failures")
    df_rejected = df.filter(F.size("validation_failures") >  0)

    # 6. Write Silver (clean)
    (df_clean.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TABLE))
    rows_clean = spark.table(SILVER_TABLE).count()

    # 7. Write rejected_records (failed rows + their failure reasons)
    (df_rejected.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(REJECTED_TABLE))
    rows_rejected = spark.table(REJECTED_TABLE).count()

    # 8. Audit log — rows_out = clean only; rejected count is derivable from the rejected table
    log_pipeline_run("silver", rows_in, rows_clean, "SUCCESS")
    print(f"Quarantined {rows_rejected:,} rows to {REJECTED_TABLE}")

except Exception as e:
    log_pipeline_run("silver", -1, 0, "FAILED", str(e))
    raise

/home/spark-0031a8a9-2527-4533-a7a0-79/.ipykernel/2137/command-7449222514757028-4272143697:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  [(PIPELINE_NAME, RUN_ID, stage, rows_in, rows_out, status, error_message, datetime.utcnow())],


[silver] SUCCESS | rows_in=2,964,624 rows_out=2,756,127
Quarantined 68,335 rows to workspace.taxi.rejected_records


In [0]:
# Show the new shape: snake_case names + computed columns
spark.sql(f"""
    SELECT vendor_id, pickup_datetime, dropoff_datetime, passenger_count,
           trip_distance, fare_amount, trip_duration_minutes, pickup_date,
           ingestion_timestamp
    FROM {SILVER_TABLE}
    LIMIT 5
""").show(truncate=False)

# Confirm row counts at every layer
print("--- Row counts by layer ---")
print(f"Bronze: {spark.table(BRONZE_TABLE).count():,}")
print(f"Silver: {spark.table(SILVER_TABLE).count():,}")
print(f"Dropped (structural nulls): {spark.table(BRONZE_TABLE).count() - spark.table(SILVER_TABLE).count():,}")

# Audit log — should now show both bronze + silver runs
spark.sql(f"""
    SELECT stage, rows_in, rows_out, status, run_timestamp
    FROM {LOG_TABLE}
    ORDER BY run_timestamp DESC
    LIMIT 10
""").show(truncate=False)

+---------+-------------------+-------------------+---------------+-------------+-----------+---------------------+-----------+--------------------------+
|vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|fare_amount|trip_duration_minutes|pickup_date|ingestion_timestamp       |
+---------+-------------------+-------------------+---------------+-------------+-----------+---------------------+-----------+--------------------------+
|2        |2024-01-01 00:57:55|2024-01-01 01:17:43|1              |1.72         |17.7       |19.8                 |2024-01-01 |2026-05-05 10:08:09.090307|
|1        |2024-01-01 00:03:00|2024-01-01 00:09:36|1              |1.8          |10.0       |6.6                  |2024-01-01 |2026-05-05 10:08:09.090307|
|1        |2024-01-01 00:17:06|2024-01-01 00:35:01|1              |4.7          |23.3       |17.916666666666668   |2024-01-01 |2026-05-05 10:08:09.090307|
|1        |2024-01-01 00:36:38|2024-01-01 00:44:56|1              |1.4

In [0]:
# How many rejected rows total?
rejected_total = spark.table(REJECTED_TABLE).count()
silver_total   = spark.table(SILVER_TABLE).count()
bronze_total   = spark.table(BRONZE_TABLE).count()

print("=== Pipeline row reconciliation ===")
print(f"Bronze rows:                       {bronze_total:,}")
print(f"  - Dropped (structural nulls):    {bronze_total - silver_total - rejected_total:,}")
print(f"  - Quarantined (rule failures):   {rejected_total:,}")
print(f"  = Silver (clean):                {silver_total:,}")
assert bronze_total == (bronze_total - silver_total - rejected_total) + rejected_total + silver_total, \
    "Row counts don't reconcile — something dropped silently"
print("Reconciliation OK ✓")

# Breakdown by failure rule — explode the array, count per rule
print("\n=== Rejection breakdown by rule ===")
spark.sql(f"""
    SELECT failure_rule, COUNT(*) AS row_count
    FROM (
        SELECT explode(validation_failures) AS failure_rule
        FROM {REJECTED_TABLE}
    )
    GROUP BY failure_rule
    ORDER BY row_count DESC
""").show(truncate=False)

# Sample 5 rejected rows so you can see the pattern
print("\n=== Sample rejected rows ===")
spark.sql(f"""
    SELECT vendor_id, pickup_datetime, dropoff_datetime, passenger_count,
           trip_distance, fare_amount, validation_failures
    FROM {REJECTED_TABLE}
    LIMIT 5
""").show(truncate=False)

=== Pipeline row reconciliation ===
Bronze rows:                       2,964,624
  - Dropped (structural nulls):    140,162
  - Quarantined (rule failures):   68,335
  = Silver (clean):                2,756,127
Reconciliation OK ✓

=== Rejection breakdown by rule ===
+----------------------------+---------+
|failure_rule                |row_count|
+----------------------------+---------+
|fare_not_positive           |36225    |
|passenger_count_out_of_range|31525    |
|non_chronological_times     |759      |
+----------------------------+---------+


=== Sample rejected rows ===
+---------+-------------------+-------------------+---------------+-------------+-----------+------------------------------+
|vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|fare_amount|validation_failures           |
+---------+-------------------+-------------------+---------------+-------------+-----------+------------------------------+
|1        |2024-01-01 00:30:40|2024-01-